# 05 · AI Discourse on Hacker News

How did the HN community's relationship with AI evolve? From sceptical blog posts about "neural networks" in 2012, through the deep learning excitement of 2016, to the ChatGPT earthquake of November 2022 — quantified.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.loader import db
from src.nlp import keyword_hits, sentiment_score
from src.viz import set_style, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        score,
        comment_count,
        posted_at,
        YEAR(posted_at) AS year,
        DATE_TRUNC('month', posted_at) AS month
    FROM stories
    WHERE YEAR(posted_at) BETWEEN 2012 AND 2024
      AND score >= 1
""").df()

print(f'{len(stories):,} stories')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

4,377,057 stories


## Monthly AI story volume — the ChatGPT inflection

In [3]:
AI_TERMS = {
    'Machine Learning': [r'machine learning', r'\bml\b'],
    'Deep Learning': ['deep learning', 'neural network'],
    'GPT / OpenAI': [r'\bgpt\b', 'openai', 'gpt-4', 'gpt-3'],
    'ChatGPT': ['chatgpt'],
    'LLM': [r'\bllm\b', 'large language model'],
}

for label, patterns in AI_TERMS.items():
    stories[label] = stories['title'].apply(lambda t: keyword_hits(str(t), patterns))

monthly_totals = stories.groupby('month').size().rename('total')

fig, axes = plt.subplots(len(AI_TERMS), 1, figsize=(14, 12), sharex=True)
chatgpt_date = pd.Timestamp('2022-11-01')

for ax, (label, _) in zip(axes, AI_TERMS.items()):
    monthly = stories.groupby('month')[label].sum()
    share = (monthly / monthly_totals * 1000).fillna(0)
    ax.fill_between(share.index, share.values, alpha=0.4, color='#e8604c')
    ax.plot(share.index, share.values, color='#e8604c', linewidth=1.5)
    ax.axvline(chatgpt_date, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_ylabel(f'{label}\n(per 1k stories)', fontsize=8)
    ax.set_yticks([])

axes[0].set_title('AI discourse on Hacker News (2012–2024)', fontsize=14, fontweight='bold')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.text(0.92, 0.5, '← ChatGPT launch', va='center', rotation=90, fontsize=9, color='gray')
plt.tight_layout()
save(fig, '../data/fig_ai_discourse.png')
plt.show()

## Sentiment of AI titles over time

In [4]:
ai_stories = stories[stories[list(AI_TERMS.keys())].any(axis=1)].copy()
ai_stories['sentiment'] = ai_stories['title'].apply(sentiment_score)

sentiment_by_year = ai_stories.groupby('year')['sentiment'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e8604c' if v > 0 else '#4c6de8' for v in sentiment_by_year.values]
ax.bar(sentiment_by_year.index, sentiment_by_year.values, color=colors, edgecolor='white')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Average sentiment of AI-related HN story titles (positive = optimistic)', fontsize=13, fontweight='bold')
ax.set_ylabel('VADER compound sentiment')
plt.tight_layout()
save(fig, '../data/fig_ai_sentiment.png')
plt.show()

## Score premium: do AI stories get more upvotes?

In [5]:
stories['is_ai'] = stories[list(AI_TERMS.keys())].any(axis=1)

score_comparison = (
    stories.groupby(['year', 'is_ai'])['score']
    .median()
    .unstack()
    .rename(columns={False: 'Non-AI stories', True: 'AI stories'})
)

fig, ax = plt.subplots(figsize=(12, 4))
score_comparison.plot(ax=ax, linewidth=2, marker='o', markersize=4)
ax.set_title('Median score: AI stories vs all others', fontsize=13, fontweight='bold')
ax.set_ylabel('Median score')
plt.tight_layout()
save(fig, '../data/fig_ai_score_premium.png')
plt.show()

premium = score_comparison['AI stories'] / score_comparison['Non-AI stories']
print('Score premium of AI stories vs baseline:')
print(premium.tail(5))

Score premium of AI stories vs baseline:
year
2020    1.0
2021    0.5
2022    1.0
2023    1.0
2024    1.0
dtype: float64
